# 🌟 Demostración Práctica de DBSCAN vs K-Means

**Objetivos de esta clase:**
1. Entender por qué K-Means falla en datos con formas no esféricas.
2. Aprender los fundamentos de DBSCAN (Density-Based Spatial Clustering).
3. Visualizar el impacto de los parámetros `eps` y `min_samples`.
4. Comparar los resultados de ambos algoritmos en datasets sintéticos.

---

## 1. Importación de librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("✅ Librerías cargadas correctamente.")

## 2. Generación de datos sintéticos

Usaremos dos conjuntos de datos clásicos para demostrar las limitaciones de K-Means y la potencia de DBSCAN:

- **make_moons**: Dos medias lunas entrelazadas.
- **make_circles**: Dos anillos concéntricos.

Ambos tienen clusters con formas **no convexas** y **densidades variables**.

In [ ]:
# Parámetros comunes
n_samples = 300
noise = 0.05
random_state = 42

# Generar los datasets
X_moons, _ = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)
X_circles, _ = make_circles(n_samples=n_samples, factor=0.5, noise=noise, random_state=random_state)

print(f"Datos generados: Moons ({X_moons.shape[0]} muestras), Circles ({X_circles.shape[0]} muestras)")

## 3. Función auxiliar para graficar resultados

Esta función nos permitirá visualizar los resultados de ambos algoritmos de forma rápida y homogénea.

In [ ]:
def plot_clustering(X, labels_kmeans, labels_dbscan, title_kmeans, title_dbscan, dataset_name=""):
    """
    Dibuja dos gráficos lado a lado comparando K-Means y DBSCAN.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # K-Means
    scatter1 = ax1.scatter(X[:, 0], X[:, 1], c=labels_kmeans, cmap='viridis', s=30, alpha=0.7)
    ax1.set_title(f'K-Means – {dataset_name}\n{title_kmeans}')
    ax1.set_xlabel('Feature 1')
    ax1.set_ylabel('Feature 2')
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # DBSCAN
    # Usamos colores distintos para destacar los clusters y el ruido (-1)
    unique_labels = set(labels_dbscan)
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    scatter2 = ax2.scatter(X[:, 0], X[:, 1], c=labels_dbscan, cmap='tab10', s=30, alpha=0.7)
    ax2.set_title(f'DBSCAN – {dataset_name}\nClusters: {n_clusters} | Ruido: {sum(labels_dbscan == -1)} puntos')
    ax2.set_xlabel('Feature 1')
    ax2.set_ylabel('Feature 2')
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

## 4. Aplicar K-Means y DBSCAN a `make_moons`

**K‑Means** (con K=2) asigna cada punto a uno de los dos clusters, pero como los datos tienen forma de lunas, el resultado es pobre: corta las lunas por la mitad.

**DBSCAN**, al basarse en densidad, detecta las dos formas correctamente y no necesita especificar el número de clusters.

In [ ]:
# K-Means
kmeans = KMeans(n_clusters=2, random_state=random_state, n_init=10)
labels_kmeans_moons = kmeans.fit_predict(X_moons)

# DBSCAN
# eps: distancia máxima entre dos puntos para considerarse vecinos.
# min_samples: número mínimo de vecinos para formar un cluster denso.
dbscan = DBSCAN(eps=0.2, min_samples=5)
labels_dbscan_moons = dbscan.fit_predict(X_moons)

plot_clustering(
    X_moons,
    labels_kmeans_moons,
    labels_dbscan_moons,
    "K=2 (asume clusters esféricos)",
    "eps=0.2, min_samples=5",
    "Moons"
)

### 🔍 Interpretación de los resultados en Moons

- **K‑Means**: La línea de separación es recta (porque usa distancia euclidiana al centroide), lo que provoca que puntos de una luna se mezclen con la otra.
- **DBSCAN**: Agrupa las dos lunas de forma casi perfecta, ya que explora la densidad local. Los pocos puntos que quedan sin asignar (ruido) son aquellos en la zona de transición entre las lunas.

**Conclusión:** DBSCAN es superior cuando los clusters tienen formas curvas o irregulares.

## 5. Aplicar K-Means y DBSCAN a `make_circles`

Este dataset presenta dos anillos concéntricos. K‑Means falla porque los centroides se ubican en el centro de cada anillo, pero el anillo exterior es un "círculo hueco" que no se puede representar bien con un centroide único.

DBSCAN, nuevamente, separa los dos anillos por densidad.

In [ ]:
# K-Means
kmeans = KMeans(n_clusters=2, random_state=random_state, n_init=10)
labels_kmeans_circles = kmeans.fit_predict(X_circles)

# DBSCAN
dbscan = DBSCAN(eps=0.15, min_samples=5)
labels_dbscan_circles = dbscan.fit_predict(X_circles)

plot_clustering(
    X_circles,
    labels_kmeans_circles,
    labels_dbscan_circles,
    "K=2 (fuerza centroides)",
    "eps=0.15, min_samples=5",
    "Circles"
)

### 🔍 Interpretación de los resultados en Circles

- **K‑Means**: Al intentar minimizar la distancia al centroide, asigna puntos del anillo exterior a ambos clusters, creando una separación radial artificial.
- **DBSCAN**: Detecta el anillo interior (más denso) y el exterior (menos denso) como clusters separados. Los puntos del borde del anillo exterior pueden ser considerados ruido si la densidad baja.

**Conclusión:** DBSCAN es robusto para clusters con formas anidadas y densidades distintas.

## 6. (Opcional) Exploración de parámetros de DBSCAN

La calidad del clustering en DBSCAN depende críticamente de dos parámetros:

- **`eps`** (epsilon): radio de vecindad. Si es muy pequeño, muchos puntos quedarán como ruido; si es muy grande, los clusters se fusionarán.
- **`min_samples`**: número mínimo de vecinos para formar un núcleo. Valores altos hacen que el algoritmo sea más conservador (más ruido).

Vamos a visualizar cómo afectan estos parámetros en el dataset `make_moons`.

In [ ]:
# Probamos combinaciones de parámetros
eps_values = [0.1, 0.2, 0.3]
min_samples_values = [3, 5, 10]

fig, axes = plt.subplots(len(eps_values), len(min_samples_values), figsize=(12, 10))
fig.suptitle("Efecto de eps y min_samples en DBSCAN (Moons)", fontsize=16)

for i, eps in enumerate(eps_values):
    for j, min_s in enumerate(min_samples_values):
        db = DBSCAN(eps=eps, min_samples=min_s)
        labels = db.fit_predict(X_moons)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = sum(labels == -1)
        
        ax = axes[i, j]
        scatter = ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
        ax.set_title(f'eps={eps}, min={min_s}\nClusters={n_clusters}, Ruido={n_noise}')
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

### 📌 Lección sobre los parámetros

- **eps demasiado pequeño (0.1)** → muchos puntos son ruido, los clusters se fragmentan.
- **eps adecuado (0.2)** → se forman los dos clusters correctos con poco ruido.
- **eps demasiado grande (0.3)** → los clusters se fusionan, perdiendo la estructura.
- **min_samples** modifica la sensibilidad: valores altos aumentan el ruido pero reducen falsos clusters.

**En la práctica**, se recomienda probar varios valores y usar la métrica de Silhouette para guiar la elección, aunque la inspección visual sigue siendo muy útil en 2D.

## 7. ¿Y si los datos están en diferentes escalas? (Importancia del escalado)

DBSCAN es sensible a la escala de los datos porque la distancia euclidiana se ve afectada por las magnitudes de las features. **Siempre se debe escalar** (por ejemplo, con `StandardScaler`) antes de aplicar DBSCAN.

Para demostrarlo, generamos un dataset con una feature mucho más grande que la otra y vemos el efecto.

In [ ]:
# Datos con escala desigual
X_scaled_test = X_moons.copy()
X_scaled_test[:, 1] = X_scaled_test[:, 1] * 100  # Feature 2 amplificada

# DBSCAN sin escalar
dbscan_no_scale = DBSCAN(eps=0.2, min_samples=5)
labels_no_scale = dbscan_no_scale.fit_predict(X_scaled_test)

# DBSCAN con escalado
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X_scaled_test)
dbscan_scaled = DBSCAN(eps=0.2, min_samples=5)
labels_scaled = dbscan_scaled.fit_predict(X_normalized)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(X_scaled_test[:, 0], X_scaled_test[:, 1], c=labels_no_scale, cmap='tab10', s=30, alpha=0.7)
ax1.set_title("Sin escalar (eps=0.2) – Muy pocos vecinos")
ax1.set_xlabel("Feature 1")
ax1.set_ylabel("Feature 2 (amplificada)")

ax2.scatter(X_normalized[:, 0], X_normalized[:, 1], c=labels_scaled, cmap='tab10', s=30, alpha=0.7)
ax2.set_title("Con escalado (StandardScaler) – Clusters correctos")
ax2.set_xlabel("Feature 1 (escalada)")
ax2.set_ylabel("Feature 2 (escalada)")

plt.tight_layout()
plt.show()

### ⚠️ Conclusión sobre el escalado

- **Sin escalar**: la feature con mayor rango domina la distancia, por lo que los puntos no se agrupan correctamente (casi todo es ruido o un solo cluster).
- **Con escalado**: ambas features contribuyen equitativamente y DBSCAN recupera la estructura original.

**Regla de oro**: Siempre escalar los datos antes de usar DBSCAN (o cualquier algoritmo basado en distancias).

## 8. Resumen y Reflexión Final

- **K-Means** es rápido y funciona bien cuando los clusters son aproximadamente esféricos y de tamaño similar.
- **DBSCAN** es más flexible y robusto frente a:
  - Formas arbitrarias (no convexas).
  - Presencia de ruido (outliers).
  - Diferentes densidades (con ajuste de parámetros).
- **Inconvenientes de DBSCAN**:
  - Requiere ajustar `eps` y `min_samples`.
  - No funciona bien si las densidades varían mucho en todo el dataset.
  - Puede ser costoso computacionalmente en datasets grandes (aunque hay versiones optimizadas).


In [ ]:
from sklearn.datasets import make_blobs

# Creamos un dataset con tres clusters de formas diferentes
np.random.seed(42)
X1, _ = make_blobs(n_samples=100, centers=[[0,0]], cluster_std=0.5)
X2, _ = make_blobs(n_samples=100, centers=[[2,2]], cluster_std=0.7)
X3, _ = make_blobs(n_samples=100, centers=[[-2,2]], cluster_std=0.3)
# Añadimos ruido disperso
X_noise = np.random.uniform(-3, 4, (50, 2))

X_multishapes = np.vstack([X1, X2, X3, X_noise])

# DBSCAN
db = DBSCAN(eps=0.3, min_samples=5)
labels_multi = db.fit_predict(X_multishapes)

plt.figure(figsize=(8, 6))
plt.scatter(X_multishapes[:, 0], X_multishapes[:, 1], c=labels_multi, cmap='tab10', s=30, alpha=0.7)
plt.title("DBSCAN en multishapes (clusters y ruido)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()